In [1]:
import os
from collections import deque

In [2]:
file = 'data/day09.txt'
path = os.path.join(os.getcwd(), file)
with open(path, 'r') as fp:
    lines = [x for x in fp.readlines()]

In [3]:
def decompress(disk_map: str) -> list:
    file_id, partitions, disk_list = 0, list(disk_map), []
    while partitions:
        disk_list.extend([file_id] * int(partitions.pop(0)))
        if partitions:
            disk_list.extend([-1] * int(partitions.pop(0)))
        file_id += 1
    return disk_list


def calc_checksum(raw_map: list) -> int:
    return sum(i * j for i, j in enumerate(raw_map) if j != -1)

In [4]:
# disk_map = '2333133121414131402'
disk_map = lines[0]

In [5]:
# part_one
raw = decompress(disk_map)
gaps = [i for i, x in enumerate(raw) if x == -1]
for gap in gaps:
    while (last := raw.pop()) == -1:
        pass
    if gap > len(raw):
        raw += [last]
        break
    raw[gap] = last
print(calc_checksum(raw))

6398608069280


In [ ]:
# I rewrote this using a deque and it is a disaster. Rather than refactor this
# back to O(n^2) I would rather get good and write it in O(nlogn).

disk_init = range(len(disk_map))
disk = deque((i // 2 if not i % 2 else -1, int(disk_map[i])) for i in disk_init)
files = reversed(sorted(file for file in disk if file[0] != -1))
raw = []
for file in files:
    file_id, file_width = file
    file_index = disk.index(file)
    while disk[-1][0] == -1:
        disk.pop()
    for i in range(len(disk)):
        if disk[-1] != file:
            disk.rotate(1)
        else:
            disk.pop()
            disk.append((-1, file_width))
            disk.rotate(-i)
            break
    for i, part in enumerate(disk):
        if i > file_index:
            break
        part_id, part_width = part
        if part_id == -1 and part_width >= file_width:
            disk.rotate(-i)
            disk.popleft()
            if part_width > file_width:
                disk.appendleft((-1, part_width - file_width))
            disk.appendleft(file)
            disk.rotate(i)
            break
for part in list(disk):
    raw.extend([part[0]] * part[1])
print(calc_checksum(raw))

6427437134372
